# Trolley training comparison

This notebook trains the left-only and full rearranged datasets with identical settings. Each dataset is trained once without augmentation and once with brightness-only augmentation. Run the cells in order.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import ultralytics
import matplotlib.pyplot as plt
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

datasets_root = Path(
    r"C:\Users\johan\OneDrive - mci4me.at\Master Thesis"
    r"\Source code\YOLO Datasets\dataset"
)

left_root = datasets_root / "trolleys_left_only"
full_root = datasets_root / "trolleys"
left_yaml = left_root / "data.yaml"
full_yaml = full_root / "data.yaml"
left_project = left_root / "runs"
full_project = full_root / "runs"
comparison_dir = datasets_root / "trolley_model_comparison"

datasets = {
    "Left only": {"root": left_root, "yaml": left_yaml},
    "Full rearranged": {"root": full_root, "yaml": full_yaml},
}
image_extensions = {".jpg", ".jpeg", ".png"}

# Refuse to train if any split is missing an image or label.
for dataset_name, config in datasets.items():
    if not config["yaml"].is_file():
        raise FileNotFoundError(config["yaml"])
    for split in ("train", "val"):
        image_dir = config["root"] / "images" / split
        label_dir = config["root"] / "labels" / split
        if not image_dir.is_dir() or not label_dir.is_dir():
            raise FileNotFoundError(f"Missing split folders for {dataset_name}/{split}")
        images = sorted(
            path for path in image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in image_extensions
        )
        labels = sorted(label_dir.glob("*.txt"))
        missing_labels = [
            image.name for image in images
            if not (label_dir / f"{image.stem}.txt").is_file()
        ]
        image_stems = {image.stem.lower() for image in images}
        orphan_labels = [
            label.name for label in labels
            if label.stem.lower() not in image_stems
        ]
        if missing_labels or orphan_labels:
            raise RuntimeError(
                f"{dataset_name}/{split} is inconsistent: "
                f"{len(missing_labels)} missing labels, {len(orphan_labels)} orphan labels."
            )
        print(f"{dataset_name:<16} {split:<5}: {len(images):>4} paired images")

device = 0 if torch.cuda.is_available() else "cpu"
batch = 0.70 if torch.cuda.is_available() else 8

print(f"\nUltralytics: {ultralytics.__version__}")
print(f"PyTorch:     {torch.__version__}")
print(f"Device:      {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
common_training = dict(
    epochs=200,
    patience=35,
    imgsz=640,
    batch=batch,
    device=device,
    workers=0,  # Reliable in Windows/Jupyter
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    cos_lr=True,
    pretrained=True,
    amp=True,
    cache="disk",
    seed=SEED,
    deterministic=True,
    plots=True,
    save=True,
    save_period=10,
    val=True,
    exist_ok=True,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=0,
)

def train_experiment(data_yaml, project, run_name, brightness):
    model = YOLO("yolov8s.pt")
    model.train(
        **common_training,
        data=str(data_yaml),
        project=str(project),
        name=run_name,
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=brightness,
    )
    best = project / run_name / "weights" / "best.pt"
    if not best.is_file():
        raise FileNotFoundError(best)
    return best


In [ ]:
# Left-camera dataset: baseline and brightness-only augmentation
left_baseline_best = train_experiment(
    left_yaml, left_project, "yolov8s_no_augmentation", brightness=0.0
)
left_brightness_best = train_experiment(
    left_yaml, left_project, "yolov8s_brightness_augmentation", brightness=0.40
)

print(f"Left baseline:   {left_baseline_best}")
print(f"Left brightness: {left_brightness_best}")

In [ ]:
# Full rearranged dataset: same model and settings for a controlled comparison
full_baseline_best = train_experiment(
    full_yaml, full_project, "yolov8s_full_no_augmentation", brightness=0.0
)
full_brightness_best = train_experiment(
    full_yaml, full_project, "yolov8s_full_brightness_augmentation", brightness=0.40
)

print(f"Full baseline:   {full_baseline_best}")
print(f"Full brightness: {full_brightness_best}")

In [ ]:
experiments = [
    {
        "Dataset": "Left only",
        "Augmentation": "None",
        "weights": left_baseline_best,
        "yaml": left_yaml,
        "run_name": "left_no_augmentation",
    },
    {
        "Dataset": "Left only",
        "Augmentation": "Brightness",
        "weights": left_brightness_best,
        "yaml": left_yaml,
        "run_name": "left_brightness",
    },
    {
        "Dataset": "Full rearranged",
        "Augmentation": "None",
        "weights": full_baseline_best,
        "yaml": full_yaml,
        "run_name": "full_no_augmentation",
    },
    {
        "Dataset": "Full rearranged",
        "Augmentation": "Brightness",
        "weights": full_brightness_best,
        "yaml": full_yaml,
        "run_name": "full_brightness",
    },
]

comparison_dir.mkdir(parents=True, exist_ok=True)
comparison_rows = []

for experiment in experiments:
    weights = experiment["weights"]
    if not weights.is_file():
        raise FileNotFoundError(weights)

    metrics = YOLO(str(weights)).val(
        data=str(experiment["yaml"]),
        split="val",
        imgsz=640,
        batch=batch,
        device=device,
        workers=0,
        plots=True,
        project=str(comparison_dir),
        name=experiment["run_name"],
        exist_ok=True,
    )

    comparison_rows.append({
        "Dataset": experiment["Dataset"],
        "Augmentation": experiment["Augmentation"],
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
    })

comparison = pd.DataFrame(comparison_rows)
comparison = comparison.sort_values(
    ["Dataset", "mAP50-95"], ascending=[True, False]
).reset_index(drop=True)

display(
    comparison.style
    .format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "mAP50": "{:.4f}",
        "mAP50-95": "{:.4f}",
    })
    .highlight_max(
        subset=["Precision", "Recall", "mAP50", "mAP50-95"],
        color="lightgreen",
    )
)

comparison.to_csv(comparison_dir / "model_comparison.csv", index=False)
print("Compare augmentation variants within each dataset first.")
print("Cross-dataset scores use different validation images and are therefore descriptive only.")

In [ ]:
run_paths = {
    "Left / no augmentation": left_project / "yolov8s_no_augmentation" / "results.csv",
    "Left / brightness": left_project / "yolov8s_brightness_augmentation" / "results.csv",
    "Full / no augmentation": full_project / "yolov8s_full_no_augmentation" / "results.csv",
    "Full / brightness": full_project / "yolov8s_full_brightness_augmentation" / "results.csv",
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for name, csv_path in run_paths.items():
    if not csv_path.is_file():
        raise FileNotFoundError(csv_path)
    results = pd.read_csv(csv_path)
    results.columns = results.columns.str.strip()
    axes[0].plot(results["epoch"], results["metrics/mAP50(B)"], label=name)
    axes[1].plot(results["epoch"], results["metrics/mAP50-95(B)"], label=name)

axes[0].set_title("Validation mAP50")
axes[1].set_title("Validation mAP50-95")
for axis in axes:
    axis.set_xlabel("Epoch")
    axis.set_ylabel("Score")
    axis.grid(alpha=0.3)
    axis.legend()

plt.tight_layout()
plt.show()

In [2]:
import sys
import platform
import torch
import ultralytics

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("Operating system:", platform.platform())
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Training device: CPU")

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PyTorch: 2.10.0+cpu
Ultralytics: 8.4.14
Operating system: Windows-10-10.0.26200-SP0
CUDA available: False
PyTorch CUDA version: None
cuDNN version: None
Training device: CPU
